# Bibliotheken und Einstellungen
Diese Zelle zuerst ausführen. Alle Imports stehen gesammelt am Anfang.

In [ ]:

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
from IPython.display import display
from scipy.stats import randint, loguniform, uniform

from IPython.display import display, HTML
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_validate,
    GridSearchCV,
    RandomizedSearchCV,
)
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")
print("Imports abgeschlossen. Noch kein Training durchgeführt.")


# CSV einfach laden

In [ ]:
CSV_PATH = Path(
    "C:/Users/mpero/Desktop/Machine Learning/Supervised/Projekt/"
    "VSCode/data/big-five-personality/data/data_clean.csv"
)

if not CSV_PATH.is_file():
    raise FileNotFoundError(f"CSV nicht gefunden. Bitte CSV_PATH prüfen: {CSV_PATH}")

df = pd.read_csv(CSV_PATH)

print("Geladene Datei:", CSV_PATH.name)
print(f"Dimensionen: {df.shape[0]:,} Zeilen, {df.shape[1]} Spalten")
print("Spalten:", df.columns.tolist())
display(df.head(5))


# Definition von X und y

In [ ]:
required_columns = {
    "N6", "N8", "N7", "N1", "N9", "N10", "N3", "N5",
    "E3", "N2", "E5", "N4", "E7", "E4", "age", "C4",
    "E1", "A4", "E10", "E9", "gender", "hand", "target",
}
missing_columns = required_columns - set(df.columns)
if missing_columns:
    raise ValueError(f"Benötigte Spalten fehlen: {sorted(missing_columns)}")
if df["target"].isna().any():
    raise ValueError("target enthält fehlende Werte. Bitte vor dem Training klären.")

X = df.drop(columns=["target"]).copy()
y = df["target"].copy()

# gender und hand sind Kategorien, auch wenn sie als Zahlen codiert wären.
categorical_features = [
    column for column in X.columns
    if column in {"gender", "hand"}
    or not pd.api.types.is_numeric_dtype(X[column])
]
numeric_features = [
    column for column in X.columns if column not in categorical_features
]

print("Numerische Merkmale:", numeric_features)
print("Kategoriale Merkmale:", categorical_features)
print("Target: target")


# Vorverarbeitung definieren
Numerische Merkmale: fehlende Werte werden mit dem Median ersetzt und standardisiert. 

Kategorien: der häufigste Wert wird einsetzt und One-Hot-Encoding verwendet. 

In [ ]:
# Hier werden nur die Schritte definiert, noch keine Werte gelernt.
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
], remainder="drop", sparse_threshold=0)

print("Vorverarbeitung definiert. Noch kein fit ausgeführt.")


# Vier Modell-Pipelines erstellen
Jede Pipeline verbindet Vorverarbeitung und Modell. Jede eigene Kopie verhindert, dass die Pipelines einen gemeinsam trainierten Preprocessor teilen. Welches Modell gewinnt, entscheiden später die CV-Ergebnisse.

In [ ]:
# Jede Pipeline erhält eine eigene Kopie des Preprocessors.
pipe_logreg = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", LogisticRegression(max_iter=2000, random_state=42)),
])

pipe_knn = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", KNeighborsClassifier()),
])

pipe_rf = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=42)),
])

pipe_hgb = Pipeline([
    ("preprocessor", clone(preprocessor)),
    ("model", HistGradientBoostingClassifier(random_state=42)),
])

# Gemeinsame Sammlung für späteren Vergleich und Hyperparameter-Suche.
pipelines = {
    "pipe_logreg": pipe_logreg,
    "pipe_knn": pipe_knn,
    "pipe_rf": pipe_rf,
    "pipe_hgb": pipe_hgb,
}

pipeline_overview = pd.DataFrame([
    {
        "Pipeline": "pipe_logreg",
        "Modell": "LogisticRegression",
        "Familie": "Lineares Modell",
        "Warum?": "Einfacher, gut interpretierbarer Ausgangspunkt.",
    },
    {
        "Pipeline": "pipe_knn",
        "Modell": "KNeighborsClassifier",
        "Familie": "Distanzbasiert",
        "Warum?": "Klassifiziert anhand ähnlicher Antworten; Skalierung ist wichtig.",
    },
    {
        "Pipeline": "pipe_rf",
        "Modell": "RandomForestClassifier",
        "Familie": "Bagging / Entscheidungsbäume",
        "Warum?": "Erfasst nichtlineare Zusammenhänge und Merkmalskombinationen.",
    },
    {
        "Pipeline": "pipe_hgb",
        "Modell": "HistGradientBoostingClassifier",
        "Familie": "Boosting / Entscheidungsbäume",
        "Warum?": "Histogrammbasiertes Boosting als weiterer Kandidat für Tabellendaten.",
    },
])

# HTML-Anzeige zur besseren Übersicht.
table_css = '''
<style>
table.pipeline-table { width: 100%; table-layout: fixed; border-collapse: collapse; }
table.pipeline-table th, table.pipeline-table td {
    white-space: normal; overflow-wrap: anywhere;
    text-align: left; vertical-align: top; padding: 8px;
}
table.pipeline-table th:nth-child(1) { width: 15%; }
table.pipeline-table th:nth-child(2) { width: 25%; }
table.pipeline-table th:nth-child(3) { width: 20%; }
table.pipeline-table th:nth-child(4) { width: 40%; }
</style>
'''

table_html = pipeline_overview.to_html(index=False, classes="pipeline-table", border=0)
display(HTML(table_css + table_html))
print("Vier Pipelines erstellt. Noch kein Training und noch kein Gewinner.")


# Train-Test-Split
Wir reservieren 20 % der Daten als Testset. test_size=0.20 bestimmt diesen Anteil, stratify=y erhält ungefähr dieselben Klassenanteile. Alle vier Modelle verwenden dieselbe Aufteilung. Die Testdaten bleiben bis zur finalen Bewertung unberührt.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Trainingsdaten:", X_train.shape)
print("Testdaten:", X_test.shape)


# Cross-Validation definieren
n_splits=5 legt fünf Durchläufe fest. shuffle=True mischt die Zeilen vor der Aufteilung; random_state=42 macht diese wiederholbar. StratifiedKFold berücksichtigt die Klassenanteile.

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


# Die Vier Pipelines trainieren und vergleichen
Die Schleife führt für jede Pipeline fünf CV-Trainings auf X_train und y_train aus: insgesamt 20 Trainingsläufe. Die Vorverarbeitung wird in jedem Durchlauf nur auf dessen Trainingsanteil gelernt.

Train F1-Macro: Wie gut erkennt das Modell die Klassen bei den Daten, mit denen es gelernt hat?

CV F1-Macro: Wie gut erkennt es die Klassen bei Daten, die es im jeweiligen Durchlauf noch nicht gesehen hat? Der Durchschnitt aus fünf Durchläufen ist unsere wichtigste Vergleichszahl.

CV F1-Streuung: Wie unterschiedlich sind die Ergebnisse der fünf Durchläufe? Eine kleine Streuung bedeutet gleichmäßigere Ergebnisse.

CV Accuracy: Welcher Anteil der Vorhersagen war richtig? 0,80 bedeutet beispielsweise 80 % richtige Vorhersagen.

In [ ]:
results = []
timing_results = []

model_names = {
    "pipe_logreg": "LogisticRegression",
    "pipe_knn": "KNN",
    "pipe_rf": "RandomForest",
    "pipe_hgb": "HistGradientBoosting"
}

for name, pipeline in pipelines.items():
    model_name = model_names[name]
    print(f"Trainiere {model_name} ...", flush=True)

    start = time.time()

    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring={
            "f1_macro": "f1_macro",
            "accuracy": "accuracy"
        },
        return_train_score=True
    )

    duration = time.time() - start
    timing_results.append({
        "Modell": model_name,
        "CV-Dauer (Sekunden)": duration
    })

    results.append({
        "Modell": model_name,
        "Train F1-Macro": scores["train_f1_macro"].mean(),
        "CV F1-Macro": scores["test_f1_macro"].mean(),
        "CV F1-Streuung": scores["test_f1_macro"].std(),
        "CV Accuracy": scores["test_accuracy"].mean()
    })

# Darstellung in HTML zur besseren Übesicht.
results_df = pd.DataFrame(results)
results_df = results_df.sort_values("CV F1-Macro", ascending=False).reset_index(drop=True)
display(HTML(results_df.round(4).to_html(index=False, border=0)))

# Zeit für alle fünf CV-Durchläufe je Pipeline, inklusive Vorverarbeitung.
timing_df = pd.DataFrame(timing_results)
print("Laufzeiten der Cross-Validation:")
display(HTML(timing_df.round(2).to_html(index=False, border=0)))
